# 05 — Agentic AI, With LangChain
Same two tools, same task, now via LangChain's `Tool` + agent executor. Compare the verbosity/tracing to your hand-rolled loop in notebook 04.

**Corrected in this version:** `InHouseLLM(...)` replaced with `get_chat_model(...)`.

# Setup
Run this first in every notebook. It assumes this notebook lives in a folder
that can reach `inhouse_wrappers.py` (the CORRECTED version, from `wrapper_fix/`),
`rag_pure_python.py`, and `inhouse_llm.py`. Adjust the `sys.path.append(...)`
lines below if your folder layout differs.

**Corrected in this version:** uses `ask()`/`ask_vision()` (built on the fixed
`get_chat_model()`) instead of calling `multimodal_chat()` directly — the
original always hit the Qwen3-14B endpoint regardless of which `model=` you
asked for. Embeddings go through `embedder.embed_query()`/`.embed_documents()`
instead of `get_embedding(text, model=MODEL_JINA)`, which doesn't match the
real function signature in your `inhouse_llm.py` (no `model=` kwarg there).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../wrapper_fix"))           # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("."))                          # folder containing inhouse_llm.py / rag_pure_python.py
# sys.path.append("/path/to/inhouse_rag_capstone")              # uncomment & adjust if needed

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

In [ ]:
# pip install langchain langchain-core --break-system-packages
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

## 1. Wrap the same Python functions as LangChain Tools
**Why:** `Tool` just adds a name/description LangChain uses to build the agent prompt automatically — the underlying function is identical to notebook 04.

In [ ]:
def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

def knowledge_lookup(query: str) -> str:
    facts = [
        "MCP standardizes how LLMs call external tools through a client-server interface.",
        "RAG combines a retriever and a generator to ground LLM answers in retrieved context.",
        "An agent is an LLM that chooses actions in a loop based on observations.",
    ]
    store = SimpleVectorStore()
    store.add(facts)
    return store.search(query, k=1)[0][0]

tools = [
    Tool(name="calculator", func=calculator, description="Evaluates a math expression."),
    Tool(name="knowledge_lookup", func=knowledge_lookup, description="Looks up a fact about RAG/MCP/agents."),
]

## 2. Build the agent
LangChain's ReAct prompt is more verbose than our hand-written one in 04 — it includes formatting instructions tuned for models that aren't great at terse JSON. **When LangChain's agent helps:** when you want a battle-tested prompt format and automatic parsing/retry logic instead of writing your own.

In [ ]:
react_prompt = PromptTemplate.from_template("""Answer the following question as best you can.
You have access to these tools:

{tools}

Use this format:
Question: the input question
Thought: reason about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer

Question: {input}
{agent_scratchpad}""")

agent_llm = get_chat_model(model=MODEL_QWEN3_30B, max_tokens=300)
agent = create_react_agent(agent_llm, tools, react_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5,
                          handle_parsing_errors=True)

result = executor.invoke({"input": "What is MCP, and what is 12 * 7?"})
print("\nFINAL ANSWER:", result["output"])

### Compare to notebook 04
- Manual loop: you control the exact prompt/parse format, easy to debug, but you wrote all of it.
- LangChain agent: more robust to small parsing slips (`handle_parsing_errors=True`), verbose tracing built in, but the actual prompt LangChain sends is longer than the JSON-only version — worth comparing token usage if you're running smaller models.

**Next step beyond this notebook:** LangGraph, for agents with branching/cyclical logic more complex than a single ReAct loop.